# Host Bucket Analysis — `host_bucket=0000.parquet`

This file is one of 10,000 produced by reorganizing the CC-MAIN-2025-26 host-bucket shards.
Each file contains **all pages from hosts whose `xxhash(hostname) % 10000 == N`**, sorted by `url_host_name`.

**Key property**: every page from `scratch.mit.edu` is in the same file, contiguous rows — ready for DBSCAN layout clustering without any cross-file shuffling.

This notebook answers:
1. How many hosts and pages are in this bucket?
2. What is the distribution of pages per host?
3. What languages and content types are present?
4. Is the hostname locality guarantee holding? (all rows for a host are contiguous)
5. What does a sample of the actual URLs look like?

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

matplotlib.rcParams["figure.dpi"] = 100

PATH = "/raid/vjawa/dripper_tutorial/host_bucket_0000.parquet"


def read_parquet(path):
    return pq.ParquetFile(path).read().to_pandas()


df = read_parquet(PATH)
print(f"Rows:    {len(df):,}")
print(f"Columns: {list(df.columns)}")
print()
print(df.dtypes)

## 1. Top-level counts

In [ ]:
n_hosts = df["url_host_name"].nunique()
n_urls = df["url"].nunique()
dup_rows = len(df) - n_urls

print(f"Total rows (pages):      {len(df):>10,}")
print(f"Unique hostnames:        {n_hosts:>10,}")
print(f"Unique URLs:             {n_urls:>10,}")
print(f"Duplicate URLs:          {dup_rows:>10,}  ({dup_rows / len(df) * 100:.3f}%)")
print(f"Avg pages / host:        {len(df) / n_hosts:>10.1f}")
print(f"Median pages / host:     {df['url_host_name'].value_counts().median():>10.0f}")

## 2. Pages-per-host distribution

In [ ]:
vc = df["url_host_name"].value_counts()

print("Pages per host — percentiles:")
for p in [50, 75, 90, 95, 99, 99.9, 100]:
    print(f"  p{p:>5.1f}: {np.percentile(vc, p):>8.0f} pages")
print()
print(f"Hosts with 1 page:      {(vc == 1).sum():>8,}  ({(vc == 1).sum() / n_hosts * 100:.1f}%)")
print(
    f"Hosts with 2-9 pages:   {((vc >= 2) & (vc < 10)).sum():>8,}  ({((vc >= 2) & (vc < 10)).sum() / n_hosts * 100:.1f}%)"
)
print(
    f"Hosts with 10-99 pages: {((vc >= 10) & (vc < 100)).sum():>8,}  ({((vc >= 10) & (vc < 100)).sum() / n_hosts * 100:.1f}%)"
)
print(f"Hosts with 100+ pages:  {(vc >= 100).sum():>8,}  ({(vc >= 100).sum() / n_hosts * 100:.1f}%)")
print(f"Hosts with 1000+ pages: {(vc >= 1000).sum():>8,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Log-scale histogram of pages per host
axes[0].hist(vc, bins=50, log=True, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Pages per host")
axes[0].set_ylabel("Number of hosts (log scale)")
axes[0].set_title("Distribution: pages per host")
axes[0].set_xscale("log")

# Cumulative: % of pages covered by top-N hosts
sorted_counts = vc.sort_values(ascending=False).values
cumulative = np.cumsum(sorted_counts) / len(df) * 100
x = np.arange(1, len(cumulative) + 1)
axes[1].plot(x, cumulative, color="orange", linewidth=1.5)
axes[1].axhline(50, color="gray", linestyle="--", alpha=0.5, label="50%")
axes[1].axhline(80, color="gray", linestyle=":", alpha=0.5, label="80%")
axes[1].set_xscale("log")
axes[1].set_xlabel("Top N hosts (log scale)")
axes[1].set_ylabel("% of total pages covered")
axes[1].set_title("Cumulative page coverage by top hosts")
axes[1].legend()
axes[1].set_ylim(0, 105)

# Annotate how many hosts cover 50% and 80%
for pct in [50, 80]:
    idx = np.searchsorted(cumulative, pct)
    axes[1].annotate(
        f"{idx:,} hosts\ncover {pct}%",
        xy=(idx, pct),
        xytext=(idx * 3, pct - 12),
        fontsize=8,
        arrowprops={"arrowstyle"="->", "color": "gray"},
    )

plt.tight_layout()
plt.show()

## 3. Top hosts

In [ ]:
print("Top 25 hosts by page count:")
print(vc.head(25).to_string())

## 4. Language distribution

In [ ]:
if "content_languages" in df.columns:
    lang_vc = df["content_languages"].fillna("unknown").value_counts()
    print(f"Unique languages: {len(lang_vc)}")
    print()
    print("Top 20 languages:")
    print(lang_vc.head(20).to_string())
    print()
    # Pie chart of top languages
    top_langs = lang_vc.head(10)
    other = lang_vc.iloc[10:].sum()
    pie_data = pd.concat([top_langs, pd.Series({"other": other})])
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.pie(pie_data, labels=pie_data.index, autopct="%1.1f%%", startangle=90)
    ax.set_title("Language distribution in host_bucket=0000")
    plt.tight_layout()
    plt.show()

## 5. Content type distribution

In [ ]:
if "content_mime_detected" in df.columns:
    mime_vc = df["content_mime_detected"].fillna("unknown").value_counts()
    print("Content MIME types (detected):")
    print(mime_vc.head(15).to_string())
    print()
    # Are all HTML?
    html_pct = mime_vc.get("text/html", 0) / len(df) * 100
    print(f"HTML pages: {html_pct:.1f}%")
    print(f"Non-HTML:   {100 - html_pct:.1f}%  (will be skipped by Dripper)")

## 6. Hostname locality check

Since we sorted by `url_host_name`, all rows for a given host should be contiguous (no interleaving). This is the key property that allows DBSCAN to run efficiently — we can stream one host at a time without random access.

In [ ]:
# Check: how many times does the hostname change across consecutive rows?
host_changes = (df["url_host_name"] != df["url_host_name"].shift()).sum() - 1  # -1 for first row

print(f"Total rows:              {len(df):,}")
print(f"Unique hosts:            {n_hosts:,}")
print(f"Host transitions in file:{host_changes:,}")
print()
if host_changes == n_hosts - 1:
    print("✅ PERFECT locality — each host appears as exactly one contiguous block")
    print("   (host transitions == unique_hosts - 1)")
elif host_changes < n_hosts * 1.01:
    extra = host_changes - (n_hosts - 1)
    print(f"✅ Near-perfect locality — {extra} hosts have a minor split ({extra / n_hosts * 100:.4f}%)")
else:
    print(f"⚠ Locality not guaranteed — {host_changes - (n_hosts - 1)} extra transitions")

# Show the actual split hosts if any
host_run_counts = (
    df.groupby((df["url_host_name"] != df["url_host_name"].shift()).cumsum())["url_host_name"].first().value_counts()
)
split_hosts = host_run_counts[host_run_counts > 1]
if len(split_hosts):
    print("\nHosts with split blocks:")
    print(split_hosts.to_string())
else:
    print("\nNo split hosts found.")

## 7. Sample URLs from interesting hosts

In [ ]:
# Show the top 5 hosts and sample URLs from each
for host in vc.head(5).index:
    host_df = df[df["url_host_name"] == host]
    print(f"\n{host} ({len(host_df):,} pages):")
    for url in host_df["url"].sample(min(5, len(host_df)), random_state=42):
        print(f"  {url[:100]}")

## 8. WARC segment diversity

How many distinct CC crawl segments contributed to this bucket? This tells us whether a host's pages come from one WARC segment or many.

In [ ]:
if "warc_filename" in df.columns:
    # Extract segment ID from WARC filename
    df["cc_segment"] = df["warc_filename"].str.extract(r"segments/([^/]+)/")
    n_segments = df["cc_segment"].nunique()
    print(f"Distinct CC crawl segments: {n_segments}")
    print()

    # Per-host: how many segments crawled it?
    segs_per_host = df.groupby("url_host_name")["cc_segment"].nunique()
    print("Segments per host distribution:")
    print(segs_per_host.value_counts().sort_index().to_string())
    print()
    multi_seg = segs_per_host[segs_per_host > 1]
    print(f"Hosts appearing in >1 segment: {len(multi_seg):,}  ({len(multi_seg) / n_hosts * 100:.1f}%)")
    if len(multi_seg):
        print("\nTop multi-segment hosts:")
        print(multi_seg.sort_values(ascending=False).head(10).to_string())

## 9. Readiness for layout clustering

Summary of how well this bucket is set up for the Dripper layout clustering pipeline.

In [ ]:
html_only = df[df.get("content_mime_detected", pd.Series(["text/html"] * len(df))) == "text/html"]
clusterable = vc[vc >= 2]  # hosts with ≥2 pages (min_cluster_size)
clustering_candidate_pages = df[df["url_host_name"].isin(clusterable.index)]

print("=" * 55)
print("LAYOUT CLUSTERING READINESS SUMMARY")
print("=" * 55)
print(f"Total pages:                    {len(df):>9,}")
print(f"HTML pages (Dripper-eligible):  {len(html_only):>9,}  ({len(html_only) / len(df) * 100:.1f}%)")
print()
print(f"Hosts with ≥2 pages:            {len(clusterable):>9,}  ({len(clusterable) / n_hosts * 100:.1f}% of hosts)")
print(
    f"Pages in clusterable hosts:     {clustering_candidate_pages['url'].count():>9,}  ({len(clustering_candidate_pages) / len(df) * 100:.1f}% of pages)"
)
print(f"Singleton hosts (1 page):       {(vc == 1).sum():>9,}  → standalone LLM call each")
print()
print("Theoretical max savings:")
max_savings = len(clustering_candidate_pages) - len(clusterable)
print(f"  Clusterable pages - 1 rep each = {max_savings:,} potential CPU-propagated pages")
print(f"  = {max_savings / len(df) * 100:.1f}% of total pages in this bucket")